# Figure 8 — the length-scale is a chemical claim

Same six observations, three length-scales. Too short and the posterior reverts to the mean between points, so BO degenerates towards random search; too long and it oversmooths and misses the narrow optimum.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 600)
OX = np.array([1.15, 2.90, 4.30, 6.10, 7.40, 8.90])
OY = land.f1d(OX)

settings = [(0.18, "too short", "reverts to the mean between points"),
            (0.85, "about right", "interpolates, and extrapolates honestly"),
            (4.00, "too long", "oversmooths — the optimum disappears")]

fig, axes = plt.subplots(1, 3, figsize=(style.FIG_W_FULL, 2.9), sharey=True,
                         gridspec_kw=dict(wspace=0.07))
for ax, (ls, tag, note) in zip(axes, settings):
    g = gpmod.GP(gpmod.matern52, ls=ls, sf=1.0, sn=0.03).fit(OX[:, None], OY)
    mu, sd = g.predict(xs[:, None])
    ax.plot(xs, land.f1d(xs), "--", color=style.INK, lw=1.0, alpha=0.45)
    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL,
                    alpha=0.16, lw=0)
    ax.plot(xs, mu, color=style.TEAL, lw=1.9)
    ax.plot(OX, OY, "o", ms=6, color=style.RED, mec="white", mew=1.0, zorder=6)
    ax.set_ylim(-2.4, 2.6)
    ax.set_xlabel("reaction parameter  x")
    ax.set_title(fr"$\ell$ = {ls}  —  {tag}", loc="left", fontsize=10.5,
                 color=style.RED if tag != "about right" else style.INK)
    ax.text(0.03, 0.04, note, transform=ax.transAxes, fontsize=9,
            color=style.GRAY)
    print(f"ls={ls:<5} log marginal likelihood = {g.log_marginal_likelihood():8.2f}")
axes[0].set_ylabel("yield (arb.)")
axes[0].plot([], [], "--", color=style.INK, alpha=0.45, label="truth")
axes[0].legend(loc="upper left", fontsize=8.5)
style.save(fig, "fig_08_hyperparameters", OUT)